# Pipeline Bronze to Silver - Transformação e Qualidade de Dados

## Objetivo
Este notebook realiza a transformação dos dados da camada **Bronze** (dados brutos com timestamp de ingestão) para a camada **Silver** (dados limpos, traduzidos, modelados e prontos para análise).

## Estratégia de Transformação

### 1. Limpeza e Qualidade
- **Deduplicação**: Window Functions mantêm apenas o registro mais recente por chave primária
- **Padronização**: Conversão de texto para UPPER CASE (cidades, estados, nomes)
- **Tradução**: Renomeação de colunas (inglês → português) e valores (status, tipos de pagamento)
- **Tratamento de nulos**: Valores padrão para campos vazios evitam perdas em joins
- **Validações**: Filtros de qualidade (datas futuras, IDs obrigatórios)

### 2. Enriquecimento de Dados
- **Métricas calculadas**: Tempo de entrega, atrasos, indicadores de prazo
- **Conversão cambial**: BRL → USD usando cotação histórica do Banco Central
- **Forward fill**: Cotações de finais de semana preenchidas com última sexta-feira
- **Agregações**: Receita total por pedido (múltiplos pagamentos consolidados)

### 3. Modelagem Dimensional

#### Tabelas Dimensionais
| Tabela | Chave | Granularidade | Registros Esperados |
|--------|-------|---------------|---------------------|
| `dim_consumidores` | `id_consumidor` | 1 registro por cliente | ~96k |
| `dim_produtos` | `id_produto` | 1 registro por produto | ~33k |
| `dim_vendedores` | `id_vendedor` | 1 registro por seller | ~3k |
| `dim_categoria_produtos_traducao` | `nome_produto_pt` | 1 registro por categoria | 71 |
| `dim_cotacao_dolar` | `data_cotacao` | 1 registro por dia | ~850 |

#### Tabelas Fato
| Tabela | Granularidade | Relacionamentos | Registros Esperados |
|--------|---------------|-----------------|---------------------|
| `fat_pedidos` | 1 registro por pedido | → dim_consumidores | ~99k |
| `fat_itens_pedidos` | 1 registro por item do pedido | → dim_produtos, dim_vendedores | ~113k |
| `fat_pagamentos_pedidos` | 1 registro por parcela/método | → fat_pedidos | ~104k |
| `fat_avaliacoes_pedidos` | 1 registro por avaliação | → fat_pedidos | ~99k |
| `fat_pedido_total` | 1 registro por pedido (agregado) | → dim_cotacao_dolar | ~99k |

## Regras de Negócio Críticas

### Deduplicação
- **Critério**: Registro com `timestamp_ingestion` mais recente vence
- **Justificativa**: Dados Bronze podem ter múltiplas versões do mesmo registro
- **Técnica**: `row_number().over(Window.partitionBy("id").orderBy(F.col("timestamp_ingestion").desc()))`

### Conversão Cambial
- **Taxa**: Cotação de **compra** do dólar (BRL por 1 USD)
- **Join temporal**: `to_date(data_pedido) = to_date(data_cotacao)`
- **Finais de semana**: Usa cotação da última sexta-feira (forward fill)

### Métricas de Entrega
- **Tempo de entrega**: `datediff(data_entrega_cliente, data_compra)`
- **Atraso**: `tempo_entrega_dias - tempo_entrega_estimado_dias`
- **Entrega no prazo**: "Sim" se atraso ≤ 0, "Não" caso contrário

## Otimizações Aplicadas
- **Delta Lake**: Todas as tabelas usam formato Delta com ACID transactions
- **OPTIMIZE**: Compactação de arquivos pequenos nas tabelas fato
- **ZORDER**: Reorganização por `id_pedido` para acelerar joins
- **Overwrite Schema**: `overwriteSchema=true` permite evolução do schema

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
Catalogo_Name = "visagio"
DB_Bronze = "bronze"
DB_Silver = "silver"

# Configuração de Variáveis e Schema

## Variáveis de Ambiente

| Variável | Valor | Descrição |
|----------|-------|-------------|
| `Catalogo_Name` | `visagio` | Nome do catálogo Unity Catalog |
| `DB_Bronze` | `bronze` | Schema de origem (dados brutos) |
| `DB_Silver` | `silver` | Schema de destino (dados limpos) |

## Criação do Schema Silver
O comando `CREATE SCHEMA IF NOT EXISTS` garante que o schema Silver existe antes de iniciar as transformações.

**Idempotência**: Execuções múltiplas não geram erro se o schema já existir.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{Catalogo_Name}`.`{DB_Silver}`")
print(f"Banco de dados {Catalogo_Name}.{DB_Silver} criado/verificado com sucesso!")

Banco de dados visagio.silver criado/verificado com sucesso!


# Parte 1 - Tratamento de `tb_customers`

## Transformações Aplicadas

- **Tradução para português:** todas as colunas foram renomeadas para o padrão em português
- **Padronização em Upper Case:** campos `cidade`, `estado` e `nome_consumidor` convertidos para maiúsculas, facilitando manipulação e evitando inconsistências de case
- **Deduplicação por ID:** utilizada **Window Function** particionada por `customer_id` e ordenada de forma decrescente por `timestamp_ingestion` (campo da Bronze), garantindo que apenas o registro mais recente seja mantido
- **Remoção de timestamp:** Campo `timestamp_ingestion` é usado apenas para ordenação e não persiste na tabela Silver

In [0]:
def create_dim_consumidores(df):
    try:
        window_f = Window.partitionBy("customer_id") \
                            .orderBy(F.col("timestamp_ingestion").desc())

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_f))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("customer_id").alias("id_consumidor"),
                F.col("customer_unique_id").alias("id_consumidor_unico"),
                F.col("customer_zip_code_prefix").alias("prefixo_cep"),
                F.upper(F.col("customer_name")).alias("nome_consumidor"),
                F.upper(F.col("customer_city")).alias("cidade"),
                F.upper(F.col("customer_state")).alias("estado"),
                F.col("customer_gender").alias("genero"),
                F.col("customer_birth_date").alias("data_nascimento"),
                F.col("customer_age").alias("idade")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_consumidores"
        
        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"Tabela {target_table} atualizada com sucesso!")
    except Exception as e:
        print(f"Erro ao criar silver.dim_consumidores: {str(e)}")

create_dim_consumidores(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_customers"))

Tabela visagio.silver.dim_consumidores atualizada com sucesso!


# Parte 2 - Tratamento de `tb_orders`

## Transformações Aplicadas

- **Tradução de Status:** mapeamento completo dos status de pedidos do inglês para português
- **Cálculo de Métricas de Entrega:**
  - `tempo_entrega_dias`: diferença entre entrega ao cliente e data da compra
  - `tempo_entrega_estimado_dias`: diferença entre estimativa e data da compra
  - `diferenca_entrega_dias`: atraso ou antecipação da entrega
- **Indicador de Prazo:** campo `entrega_no_prazo` categorizando entregas (Sim/Não/Não Entregue)
- **Deduplicação por ID:** Window Function garantindo registro mais recente por `order_id`

In [0]:
def create_fat_pedidos(df):
    try:
        status_map = {
            "delivered"   : "entregue",
            "canceled"    : "cancelado",
            "shipped"     : "enviado",
            "processing"  : "processando",
            "invoiced"    : "faturado",
            "unavailable" : "indisponível",
            "created"     : "criado",
            "approved"    : "aprovado"
        }
        mapping_expr = F.create_map(
            *[F.lit(x) for pair in status_map.items() for x in pair]
        )

        df_silver = (
            df
            .withColumn("status_pt", mapping_expr[F.col("order_status")])
            .withColumn(
                "tempo_entrega_dias",
                F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
            )
            .withColumn(
                "tempo_entrega_estimado_dias",
                F.datediff(F.col("order_estimated_delivery_date"), F.col("order_purchase_timestamp"))
            )
            .withColumn(
                "diferenca_entrega_dias",
                F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
            )
            .withColumn(
                "entrega_no_prazo",
                F.when(F.col("order_status") != "delivered", "Não Entregue")
                 .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
                 .otherwise("Não")
            )
            .select(
                F.col("order_id").alias("id_pedido"),
                F.col("customer_id").alias("id_consumidor"),
                F.col("status_pt").alias("status"),
                F.col("order_purchase_timestamp").alias("data_compra"),
                F.col("order_approved_at").alias("data_aprovacao"),
                F.col("order_delivered_carrier_date").alias("data_entrega_transportadora"),
                F.col("order_delivered_customer_date").alias("data_entrega_cliente"),
                F.col("order_estimated_delivery_date").alias("data_estimada_entrega"),
                F.col("tempo_entrega_dias"),
                F.col("tempo_entrega_estimado_dias"),
                F.col("diferenca_entrega_dias"),
                F.col("entrega_no_prazo")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_pedidos"

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso!")

    except Exception as e:
        print(f"❌ Erro ao criar silver.fat_pedidos: {str(e)}")

create_fat_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_orders"))

✅ Tabela visagio.silver.fat_pedidos atualizada com sucesso!


# Parte 3 - Tratamento de `tb_order_items`

## Objetivo
Criar tabela fato com granularidade de **item por pedido**, preservando detalhes de cada produto vendido.

## Transformações Aplicadas

### 1. Deduplicação
- **Chave**: Combinação de `order_id` + `order_item_id` (chave composta)
- **Window Function**: `partitionBy("order_id", "order_item_id").orderBy(timestamp_ingestion.desc())`
- **Critério**: Mantém apenas o registro com timestamp mais recente
- **Justificativa**: Evita duplicatas de itens que podem ter sido reimportados ou corrigidos na Bronze

### 2. Tradução de Colunas
| Coluna Bronze | Coluna Silver | Tipo |
|---------------|---------------|------|
| `order_id` | `id_pedido` | string |
| `order_item_id` | `id_item` | int |
| `product_id` | `id_produto` | string |
| `seller_id` | `id_vendedor` | string |
| `shipping_limit_date` | `data_limite_envio` | timestamp |
| `price` | `preco_BRL` | double |
| `freight_value` | `preco_frete` | double |

### 3. Granularidade
- **Nível**: Item individual dentro de cada pedido
- **Relação com fat_pedidos**: 1:N (um pedido pode ter vários itens)
- **Uso**: Análises de mix de produtos, preços por item, frete individual

### 4. Preços Separados
- **preco_BRL**: Preço do produto (sem frete)
- **preco_frete**: Valor do frete para este item específico
- **Justificativa**: Permite análises separadas de margem vs custo logístico

## Por Que Não Há Agregação?
Esta é uma tabela **fato granular** (nível item). Para receita total por pedido, use `fat_pedido_total`.

In [0]:
def create_fat_itens_pedidos(df):
    try:
        window_spec = Window.partitionBy("order_id", "order_item_id") \
                            .orderBy(F.col("timestamp_ingestion").desc())

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("order_id").alias("id_pedido"),
                F.col("order_item_id").alias("id_item"),
                F.col("product_id").alias("id_produto"),
                F.col("seller_id").alias("id_vendedor"),
                F.col("shipping_limit_date").alias("data_limite_envio"), 
                F.col("price").alias("preco_BRL"),
                F.col("freight_value").alias("preco_frete")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_itens_pedidos"

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"Tabela {target_table} atualizada com sucesso!")

    except Exception as e:
        print(f"Erro ao criar silver.fat_itens_pedidos: {str(e)}")

create_fat_itens_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_order_items"))

Tabela visagio.silver.fat_itens_pedidos atualizada com sucesso!


# Parte 4 - Tratamento de `tb_order_payments`

## Objetivo
Criar tabela fato com granularidade de **método de pagamento por pedido**, capturando parcelamentos e múltiplos meios de pagamento.

## Transformações Aplicadas

### 1. Deduplicação
- **Chave composta**: `order_id` + `payment_sequential`
- **payment_sequential**: Número sequencial do pagamento (1, 2, 3... quando há múltiplos métodos)
- **Window Function**: `partitionBy("order_id", "payment_sequential").orderBy(timestamp_ingestion.desc())`
- **Justificativa**: Alguns pedidos usam cartão + boleto ou têm várias tentativas de pagamento

### 2. Tradução de Métodos de Pagamento

| Valor Bronze (EN) | Valor Silver (PT) | Descrição |
|-------------------|-------------------|-------------|
| `credit_card` | Cartão de Crédito | Pagamento via cartão |
| `boleto` | Boleto | Boleto bancário |
| `voucher` | Voucher | Voucher/cupom |
| `debit_card` | Cartão de Débito | Pagamento via débito |
| `not_defined` | Não Definido | Método não especificado |

### 3. Tradução de Colunas
| Coluna Bronze | Coluna Silver | Tipo |
|---------------|---------------|------|
| `order_id` | `id_pedido` | string |
| `payment_sequential` | `sequencial_pagamento` | int |
| `payment_type` | `tipo_pagamento` | string (traduzido) |
| `payment_installments` | `parcelas` | int |
| `payment_value` | `valor_pagamento` | double |

### 4. Granularidade de Parcelas
- **Nível**: Um registro por método/tentativa de pagamento
- **Casos de uso**:
  - Pedido pago com 1 cartão em 3x: 1 registro (sequential=1, parcelas=3)
  - Pedido pago com cartão + boleto: 2 registros (sequential=1 e sequential=2)
- **Relação com fat_pedidos**: 1:N (um pedido pode ter vários pagamentos)

### 5. Uso Esperado
- Análise de mix de pagamento (% cartão vs boleto)
- Identificação de pedidos com múltiplas tentativas de pagamento
- Cálculo de receita total: agrupar por `id_pedido` e somar `valor_pagamento` (ver `fat_pedido_total`)

In [0]:
def create_fat_pagamentos_pedidos(df):
    try:
        window_spec = Window.partitionBy("order_id", "payment_sequential") \
                            .orderBy(F.col("timestamp_ingestion").desc())

        payment_map = {
            "credit_card" : "Cartão de Crédito",
            "boleto"      : "Boleto",
            "voucher"     : "Voucher",
            "debit_card"  : "Cartão de Débito",
            "not_defined" : "Não Definido"
        }

        mapping_expr = F.create_map(
            *[F.lit(x) for pair in payment_map.items() for x in pair]
        )

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .withColumn("tipo_pagamento", mapping_expr[F.col("payment_type")])
            .select(
                F.col("order_id").alias("id_pedido"),
                F.col("payment_sequential").alias("sequencial_pagamento"),
                F.col("tipo_pagamento"),
                F.col("payment_installments").alias("parcelas"),
                F.col("payment_value").alias("valor_pagamento")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_pagamentos_pedidos"

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"Tabela {target_table} atualizada com sucesso!")

    except Exception as e:
        print(f"Erro ao criar silver.fat_pagamentos_pedidos: {str(e)}")

create_fat_pagamentos_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_order_payments"))

Tabela visagio.silver.fat_pagamentos_pedidos atualizada com sucesso!


# Parte 5 - Tratamento de `tb_order_reviews`

## Transformações Aplicadas

- **Conversão de Timestamps:** uso de `try_to_timestamp` para segurança na conversão
- **Filtros de Qualidade:**
  - Remove registros sem `order_id` (inválidos)
  - Valida timestamps futuros
- **Tratamento de Nulos:** campos vazios preenchidos com "Sem título" e "Sem comentário"
- **Tradução:** renomeação para padrão português
- **Particionamento:** `repartition(4)` para balanceamento de arquivos Delta

In [0]:
def create_fat_avaliacoes_pedidos(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_avaliacoes_pedidos"

        df_silver = (
            df
            .withColumn("review_creation_date",    F.try_to_timestamp("review_creation_date"))
            .withColumn("review_answer_timestamp", F.try_to_timestamp("review_answer_timestamp"))

            .filter(F.col("order_id").isNotNull())

            .filter(
                F.col("review_creation_date").isNull() | 
                (F.col("review_creation_date") <= F.current_timestamp())
            )

            .withColumn("review_comment_title",
                F.when(
                    F.col("review_comment_title").isNull() | (F.trim(F.col("review_comment_title")) == ""),
                    "Sem título"
                ).otherwise(F.col("review_comment_title"))
            )
            .withColumn("review_comment_message",
                F.when(
                    F.col("review_comment_message").isNull() | (F.trim(F.col("review_comment_message")) == ""),
                    "Sem comentário"
                ).otherwise(F.col("review_comment_message"))
            )

            .select(
                F.col("review_id").alias("id_avaliacao"),
                F.col("order_id").alias("id_pedido"),
                F.col("review_score").alias("nota_avaliacao"),
                F.col("review_comment_title").alias("titulo_avaliacao_comentario"),
                F.col("review_comment_message").alias("mensagem_avaliacao_comentario"),
                F.col("review_creation_date").alias("data_criacao_avaliacao"),
                F.col("review_answer_timestamp").alias("data_resposta_avaliacao")
            )
        )

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f" Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")
    except Exception as e:
        print(f" Erro ao criar silver.fat_avaliacoes_pedidos: {str(e)}")

create_fat_avaliacoes_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_order_reviews"))

 Tabela visagio.silver.fat_avaliacoes_pedidos atualizada com sucesso! (99224 registros)



# Parte 6 - Tratamento de `tb_products`

## Objetivo
Criar **dimensão de produtos** (SCD Type 1) com atributos físicos e metadados do catálogo.

## Transformações Aplicadas

### 1. Deduplicação
- **Chave**: `product_id` (única)
- **Window Function**: `partitionBy("product_id").orderBy(timestamp_ingestion.desc())`
- **Critério**: Mantém apenas a versão mais recente de cada produto
- **Justificativa**: Produtos podem ter atualizações de informações (preços, descrições, categorias)

### 2. Atributos Físicos (Logística)
Esses campos são essenciais para cálculos de frete e estoque:

| Campo | Unidade | Uso |
|-------|---------|-----|
| `peso_produto_gramas` | gramas | Cálculo de custo de transporte |
| `comprimento_centimetros` | cm | Capacidade de caixa/container |
| `altura_centimetros` | cm | Empilhamento em estoque |
| `largura_centimetros` | cm | Espaço em prateleira |

### 3. Metadados de Produto

| Campo | Tipo | Descrição |
|-------|------|-------------|
| `tamanho_nome_produto` | int | Comprimento do nome (usado para validações) |
| `tamanho_descricao_produto` | int | Comprimento da descrição |
| `quantidade_fotos` | int | Número de imagens (indicador de qualidade do anúncio) |

### 4. Relacionamento com Categoria
- **categoria_produto**: Categoria em português (pode estar nula)
- **Tradução**: Use join com `dim_categoria_produtos_traducao` para obter nome em inglês

### 5. Tipo de Tabela Dimensional
- **SCD Type 1**: Sobrescreve valores antigos (não mantém histórico)
- **Justificativa**: Atributos físicos raramente mudam; quando mudam, queremos apenas a versão atual
- **Histórico**: Se necessário rastrear mudanças, consulte a tabela Bronze com `timestamp_ingestion`

### 6. Uso Esperado
- Joins com `fat_itens_pedidos` para enriquecer análises de vendas
- Cálculos de frete por peso/dimensões
- Segmentação de produtos por categoria
- Análises de qualidade de anúncios (produtos com mais fotos vendem mais?)

In [0]:
def create_dim_produtos(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_produtos"
        window_spec = Window.partitionBy("product_id").orderBy(
            F.col("timestamp_ingestion").desc(),
        )

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("product_id").alias("id_produto"),
                F.col("product_name").alias("nome_produto"),
                F.col("product_category_name").alias("categoria_produto"),
                F.col("product_name_lenght").alias("tamanho_nome_produto"),
                F.col("product_description_lenght").alias("tamanho_descricao_produto"),
                F.col("product_photos_qty").alias("quantidade_fotos"),
                F.col("product_weight_g").alias("peso_produto_gramas"),
                F.col("product_length_cm").alias("comprimento_centimetros"),
                F.col("product_height_cm").alias("altura_centimetros"),
                F.col("product_width_cm").alias("largura_centimetros")
            )
        )

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_produtos: {str(e)}")

create_dim_produtos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_products"))

✅ Tabela visagio.silver.dim_produtos atualizada com sucesso! (32951 registros)



# Parte 7 - Tratamento de `tb_sellers`

## Objetivo
Criar **dimensão de vendedores** (SCD Type 1) com localização geográfica para análises regionais.

## Transformações Aplicadas

### 1. Deduplicação
- **Chave**: `seller_id` (única)
- **Window Function**: `partitionBy("seller_id").orderBy(timestamp_ingestion.desc())`
- **Critério**: Mantém apenas a versão mais recente de cada vendedor
- **Justificativa**: Vendedores podem mudar de endereço/razão social

### 2. Padronização Geográfica

#### Por Que UPPER CASE?
- **Problema**: Dados Bronze podem ter "São Paulo", "SAO PAULO", "são paulo"
- **Solução**: `F.upper()` em `cidade` e `estado`
- **Benefícios**:
  - Evita duplicatas por diferenças de capitalização
  - Facilita buscas e filtros (case-insensitive)
  - Padronização para dashboards e relatórios

### 3. Tradução de Colunas

| Coluna Bronze | Coluna Silver | Exemplo |
|---------------|---------------|----------|
| `seller_id` | `id_vendedor` | `abc123...` |
| `seller_zip_code_prefix` | `prefixo_cep` | `04567` (5 dígitos) |
| `seller_name` | `nome_vendedor` | `Loja XYZ Ltda` |
| `seller_city` | `cidade` | `SÃO PAULO` |
| `seller_state` | `estado` | `SP` |

### 4. Localização Regional

#### Prefixo CEP (5 dígitos)
- **Granularidade**: Define região/bairro aproximado
- **Uso**: 
  - Cálculo de distância entre vendedor e cliente
  - Análises de densidade de vendedores por região
  - Join com `tb_geolocalizacao` (Bronze) para coordenadas GPS

#### Estado (UF)
- **Uso**: Segmentação macro-regional (Sul, Sudeste, Nordeste...)
- **Análises**: Performance de vendas por estado, tempo de entrega por origem

### 5. Tipo de Tabela Dimensional
- **SCD Type 1**: Sobrescreve valores antigos
- **Justificativa**: Localização é dado mestre; quando muda, queremos apenas a versão atual

### 6. Uso Esperado
- Joins com `fat_itens_pedidos` para identificar vendedor de cada item
- Análises de concentração geográfica de sellers
- Estudos de tempo de entrega por região de origem
- Rankings de vendedores por estado/cidade

In [0]:
def create_dim_vendedores(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_vendedores"

        window_spec = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("seller_id").alias("id_vendedor"),
                F.col("seller_zip_code_prefix").alias("prefixo_cep"),
                F.col("seller_name").alias("nome_vendedor"),
                F.upper(F.col("seller_city")).alias("cidade"),
                F.upper(F.col("seller_state")).alias("estado")
            )
        )

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_vendedores: {str(e)}")

create_dim_vendedores(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_sellers"))

✅ Tabela visagio.silver.dim_vendedores atualizada com sucesso! (3095 registros)



# Parte 8 - Tratamento de `tb_product_category_name_translation`

## Objetivo
Criar **tabela de referência bilingüe** para tradução de categorias de produtos (português ↔ inglês).

## Características

### 1. Tabela Estática de Referência
- **Tipo**: Lookup table (tabela de consulta)
- **Tamanho**: 71 registros (todas as categorias do catálogo)
- **Atualização**: Raramente muda (apenas quando novas categorias são criadas)

### 2. Por Que NÃO Há Deduplicação?
- **Fonte**: Tabela de referência estática (sem versões múltiplas)
- **Chave natural**: `nome_produto_pt` (categoria em português) é única
- **Resultado**: Não há duplicatas na Bronze, logo não precisa deduplicar

### 3. Estrutura

| Coluna Silver | Tipo | Exemplo |
|---------------|------|----------|
| `nome_produto_pt` | string | `beleza_saude` |
| `nome_produto_en` | string | `health_beauty` |

### 4. Uso Esperado

#### Join com dim_produtos
```sql
SELECT 
  p.nome_produto,
  p.categoria_produto AS categoria_pt,
  t.nome_produto_en AS categoria_en
FROM visagio.silver.dim_produtos p
LEFT JOIN visagio.silver.dim_categoria_produtos_traducao t
  ON p.categoria_produto = t.nome_produto_pt
```

#### Casos de Uso
- **Dashboards bilingües**: Exibir nomes em inglês para stakeholders internacionais
- **Análises internacionais**: Padronizar nomes para comparações com outros países
- **APIs/Integrações**: Fornecer nomes traduzidos para sistemas externos

### 5. Exemplos de Tradução

| Português (PT) | Inglês (EN) |
|------------------|---------------|
| `beleza_saude` | `health_beauty` |
| `esporte_lazer` | `sports_leisure` |
| `moveis_decoracao` | `furniture_decor` |
| `informatica_acessorios` | `computers_accessories` |

### 6. Manutenção
- **Adição de categorias**: Inserir novo registro na Bronze e reexecutar pipeline
- **Correção de tradução**: Atualizar registro na Bronze e reexecutar (overwrite)

In [0]:
def create_dim_categoria_produtos_traducao(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_categoria_produtos_traducao"

        df_silver = (
            df
            .select(
                F.col("product_category_name").alias("nome_produto_pt"),
                F.col("product_category_name_english").alias("nome_produto_en")
            )
        )

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_categoria_produtos_traducao: {str(e)}")

create_dim_categoria_produtos_traducao(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_product_category_name_translation"))

✅ Tabela visagio.silver.dim_categoria_produtos_traducao atualizada com sucesso! (71 registros)



# Parte 9 - Tratamento de `tb_cotacao_dolar`

## Transformações Aplicadas

### Estratégia: Calendário Contínuo com Forward Fill

#### Problema
- API do Banco Central não retorna cotações em finais de semana/feriados
- Joins por data com `fat_pedidos` perderiam registros de pedidos nesses dias

#### Solução
1. **Calendário Contínuo**: Geração de **todas as datas** no range usando `sequence(data_min, data_max, interval 1 day)`
2. **Cotação de Fechamento**: Window Function para pegar **última cotação do dia** (algumas datas têm múltiplas cotações)
3. **Forward Fill**: Preenchimento de finais de semana/feriados com cotação da **última sexta-feira** usando `last(..., ignorenulls=True).over(window_fill)`
4. **Window Spec**: `rowsBetween(Window.unboundedPreceding, 0)` para propagação sequencial cronológica

### Por Que Não Há Deduplicação?
- **Fonte**: Dados vem direto da API do BCB (não há histórico de versões)
- **Lógica**: Já selecionamos **última cotação do dia** na Window Function de fechamento
- **Resultado**: Cada data tem **exatamente 1 registro** (dias úteis têm cotação real, finais de semana têm forward fill)

### Benefícios
- ✅ **Joins seguros**: Toda data de pedido terá uma cotação correspondente
- ✅ **Precisão**: Usa cotação do último dia útil anterior (padrão de mercado)

In [0]:
def create_dim_cotacao_dolar(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_cotacao_dolar"
        datas = df.agg(
            F.min(F.to_date("data_hora_cotacao")).alias("data_min"),
            F.max(F.to_date("data_hora_cotacao")).alias("data_max")
        ).collect()[0]

        data_min = datas["data_min"]
        data_max = datas["data_max"]

        print(f"Range: {data_min} → {data_max}")

        # Calendário contínuo — data_date como DateType para o join
        df_calendario = spark.sql(f"""
            SELECT explode(sequence(
                to_date('{data_min}'),
                to_date('{data_max}'),
                interval 1 day
            )) as data_date
        """)

        # Cotação de fechamento do dia
        window_fechamento = Window.partitionBy(F.to_date("data_hora_cotacao")).orderBy(F.col("data_hora_cotacao").desc())

        df_cotacao_fechamento = (
            df
            .withColumn("rn", F.row_number().over(window_fechamento))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .withColumn("data_date", F.to_date("data_hora_cotacao"))
        )

        df_completo = df_calendario.join(df_cotacao_fechamento, on="data_date", how="left")

        # Forward fill
        window_fill = Window.orderBy("data_date").rowsBetween(Window.unboundedPreceding, 0)

        df_silver = (
            df_completo
            .withColumn("cotacao_compra",    F.last("cotacao_compra",    ignorenulls=True).over(window_fill))
            .withColumn("data_hora_cotacao", F.last("data_hora_cotacao", ignorenulls=True).over(window_fill))
            .select(
                F.date_format(F.col("data_date"), "dd-MM-yyyy").alias("data_cotacao"),
                F.date_format(F.col("data_hora_cotacao"), "dd-MM-yyyy").alias("data_hora_cotacao"),
                F.col("cotacao_compra").alias("cotacao_compra")
            )
        )

        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_cotacao_dolar: {str(e)}")
create_dim_cotacao_dolar(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_cotacao_dolar"))

Range: 2016-09-01 → 2018-12-31


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Tabela visagio.silver.dim_cotacao_dolar atualizada com sucesso! (852 registros)



# Parte 10 - Tratamento de `fat_pedido_total`

## Objetivo
Criar uma **tabela fato agregada** consolidando pedidos + pagamentos + conversão cambial para análises de receita.

## Transformações Aplicadas

### 1. Fontes de Dados (Tabelas Silver)
- **fat_pedidos**: Data do pedido e status
- **fat_pagamentos_pedidos**: Valores pagos (pode haver múltiplas parcelas/métodos por pedido)
- **dim_cotacao_dolar**: Cotação diária para conversão BRL → USD

### 2. Agregação de Pagamentos
- **Granularidade**: Um registro por pedido (nível mais agregado que `fat_pagamentos_pedidos`)
- **Cálculo**: `SUM(valor_pagamento)` agrupado por `id_pedido`
- **Justificativa**: Pedidos podem ter múltiplos pagamentos (cartão + boleto, parcelamento)

### 3. Conversão Cambial
- **Join temporal**: Relacionamento pela `data_compra` com `dim_cotacao_dolar`
- **Cálculo**: `valor_total_pago_usd = valor_total_pago_brl / cotacao_compra`
- **Moedas duais**: Mantém BRL e USD para flexibilidade analítica
- **Precisão**: 2 casas decimais com `round()`

### 4. Por Que Não Há Deduplicação?
- **Fonte**: Tabelas Silver já deduplicadas (fat_pedidos, fat_pagamentos_pedidos)
- **Lógica**: Agregação por `id_pedido` naturalmente remove duplicatas
- **Resultado**: Um registro único por pedido (1:1)

### 5. Uso Esperado
- Dashboards de receita total (BRL e USD)
- Análises de conversão cambial
- KPIs de faturamento consolidado
- Base para cálculo de métricas Gold (ticket médio, etc.)

### 6. Particionamento
- **Otimização**: Tabela otimizada com ZORDER na célula 26 (id_pedido + data_pedido)

In [0]:
def create_fat_pedido_total():
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_pedido_total"

        # ===== LEITURA DAS TABELAS SILVER JÁ LIMPAS =====
        df_pedidos    = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_pedidos")
        df_pagamentos = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_pagamentos_pedidos")
        df_cotacao    = spark.table(f"{Catalogo_Name}.{DB_Silver}.dim_cotacao_dolar")

        # ===== AGREGAÇÃO: Soma todos os pagamentos por pedido =====
        # Regra: Um pedido pode ter múltiplos pagamentos (cartão + boleto, parcelamento)
        # Resultado: Receita total do pedido em BRL
        df_pagamentos_agg = (
            df_pagamentos
            .groupBy("id_pedido")
            .agg(F.round(F.sum("valor_pagamento"), 2).alias("valor_total_pago_brl"))
        )

        # ===== JOIN: Pedidos com pagamentos agregados =====
        # Left join garante que pedidos sem pagamento sejam mantidos (valor null)
        df_joined = (
            df_pedidos
            .join(df_pagamentos_agg, on="id_pedido", how="left")
        )

        # ===== JOIN TEMPORAL: Cotação do dólar pela data do pedido =====
        # Converte data_compra (timestamp) para date para join com dim_cotacao
        # Left join: pedidos sem cotação (erro de dados) ficam com USD null
        df_final = (
            df_joined
            .join(
                df_cotacao,
                on=F.to_date(df_joined["data_compra"]) == F.to_date(df_cotacao["data_cotacao"], "dd-MM-yyyy"),
                how="left"
            )
            .drop(df_cotacao["data_cotacao"])  # Remove coluna auxiliar após join
            # ===== CONVERSÃO CAMBIAL: BRL → USD =====
            .withColumn("valor_total_pago_usd",
                F.round(F.col("valor_total_pago_brl") / F.col("cotacao_compra"), 2)
            )
            # ===== SELEÇÃO FINAL: Campos relevantes =====
            .select(
                F.col("id_pedido"),
                F.col("id_consumidor"),
                F.col("status"),
                F.round(F.col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
                F.round(F.col("valor_total_pago_usd"), 2).alias("valor_total_pago_usd"),
                F.col("data_compra").alias("data_pedido")
            )
        )

        # ===== GRAVAÇÃO: Delta Lake =====
        df_final.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_final.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.fat_pedido_total: {str(e)}")

create_fat_pedido_total()

✅ Tabela visagio.silver.fat_pedido_total atualizada com sucesso! (99441 registros)



# Parte 11 - Otimização de Tabelas Delta

## Objetivo
Aplicar **otimizações físicas** nas tabelas fato para acelerar queries de análise.

## Operações Aplicadas

### 1. OPTIMIZE - Compactação de Arquivos

#### O Que Faz?
- **Problema**: Delta Lake cria muitos arquivos pequenos após múltiplas escritas
- **Solução**: `OPTIMIZE` mescla arquivos pequenos em arquivos maiores (target: 1 GB cada)
- **Resultado**: Menos arquivos = menos overhead de I/O = queries mais rápidas

#### Benefícios
- ✅ Reduz tempo de leitura de metadados (menos arquivos para listar)
- ✅ Melhora eficiência de cache do Spark
- ✅ Diminui latência de queries (menos arquivos para abrir)

### 2. ZORDER - Organização Física dos Dados

#### O Que Faz?
- **Técnica**: Z-order clustering (ordenação multidimensional)
- **Objetivo**: Co-localizar registros relacionados no mesmo arquivo
- **Resultado**: Data skipping mais eficiente (pula arquivos irrelevantes)

#### Colunas de Ordenação

| Tabela | Colunas ZORDER | Justificativa |
|--------|----------------|---------------|
| `fat_pedidos` | `id_pedido` | Filtros e joins frequentes por pedido |
| `fat_itens_pedidos` | `id_pedido` | Join com fat_pedidos |
| `fat_pagamentos_pedidos` | `id_pedido` | Agregações por pedido |
| `fat_pedido_total` | `id_pedido, data_pedido` | Filtros por pedido **e** período |

#### Por Que 2 Colunas em fat_pedido_total?
- **Pattern comum**: `WHERE id_pedido = X AND data_pedido BETWEEN ...`
- **ZORDER duplo**: Otimiza ambas as dimensões simultaneamente
- **Trade-off**: Menos eficiente que ZORDER simples, mas útil para queries mistas

### 3. Data Skipping (Automático)

#### Como Funciona?
- Delta Lake mantém **estatísticas de MIN/MAX por arquivo**
- ZORDER agrupa valores similares no mesmo arquivo
- Spark **pula arquivos inteiros** que não satisfazem filtros WHERE

#### Exemplo
```sql
SELECT * FROM fat_pedidos WHERE id_pedido = 'abc123'
```
- **Sem ZORDER**: Lê todos os 100 arquivos
- **Com ZORDER**: Lê apenas 1-2 arquivos que contêm 'abc123'
- **Speed-up**: 50-100x mais rápido

### 4. Tabelas Otimizadas

Apenas **tabelas fato** são otimizadas porque:
- ✅ **São grandes** (~100k registros): beneficiam de data skipping
- ✅ **Queries frequentes**: usadas em dashboards e relatórios
- ✅ **Filtros comuns**: sempre filtram por `id_pedido`

Tabelas dimensionais não precisam:
- ❌ **São pequenas** (~3k-96k registros): scan completo é rápido
- ❌ **Broadcast joins**: Spark envia inteira para workers (ignora particionação)

### 5. Quando Reexecutar?
- **Após cargas incrementais**: Quando novos dados são adicionados
- **Periodicidade sugerida**: Diário (após ETL noturno) ou semanal
- **Custo vs Benefício**: OPTIMIZE consome recursos, mas melhora todas as queries subsequentes

### 6. Monitoramento
```sql
-- Verificar quantidade de arquivos
DESCRIBE DETAIL visagio.silver.fat_pedidos;
-- Campo 'numFiles': quanto menor, melhor
```

In [0]:
tabelas_otimizar = [
    "fat_pedidos",
    "fat_itens_pedidos",
    "fat_pagamentos_pedidos",
    "fat_pedido_total",
]

for tabela in tabelas_otimizar:
    try:
        # id_pedido existe em todas, data_pedido só no fat_pedido_total
        zorder_cols = "id_pedido, data_pedido" if tabela == "fat_pedido_total" else "id_pedido"

        spark.sql(f"""
            OPTIMIZE {Catalogo_Name}.{DB_Silver}.{tabela}
            ZORDER BY ({zorder_cols})
        """)
        print(f"✅ OPTIMIZE concluído: {tabela}")

    except Exception as e:
        print(f"❌ Erro ao otimizar {tabela}: {str(e)}")

✅ OPTIMIZE concluído: fat_pedidos
✅ OPTIMIZE concluído: fat_itens_pedidos
✅ OPTIMIZE concluído: fat_pagamentos_pedidos
✅ OPTIMIZE concluído: fat_pedido_total
